<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling | Preparing Developer Survey Data for Analysis**
**IBM Skills Network guided lab | Personalized portfolio edition**


### Project overview
Before a dataset can tell us anything useful, we need to understand what is inside it and make sure the information is ready to work with. Real survey data can contain inconsistent labels, unanswered questions, and values that are difficult to compare directly.

In this notebook, I work through a practical data-wrangling process using developer survey responses. I check the dataset, standardize country labels, turn employment categories into numerical indicators, handle missing values, and create new features that could support later analysis.

**Question guiding the work:** How can we turn a large, imperfect survey dataset into a more consistent and analysis-ready starting point?

**What I'll do:** Explore the data → standardize labels → encode categories → address missing values → scale and transform compensation → group professional experience.

**Tools:** Python, Pandas, NumPy.

*This notebook is based on an IBM Skills Network guided lab. The original exercises and attribution are retained; the additional explanations are written to make the steps and their trade-offs clearer to a broader audience.*

Estimated time needed: **45** minutes


### Objectives


After completing this lab, you will be able to:
- Identify and remove inconsistent data entries.
- Encode categorical variables for analysis.
- Handle missing values using multiple imputation strategies.
- Apply feature scaling and transformation techniques.

#### Install the required libraries


In [2]:
!pip install pandas
!pip install matplotlib

### Tasks


#### Step 1: Import the necessary module.


#### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.<br>
I start by loading the survey dataset and looking at the first few rows. This quick check tells me whether the file loaded successfully and gives me a feel for the questions respondents answered. It also helps me spot fields I may want to clean later.

In [19]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


##### 2.1 Summarize the dataset by displaying the column data types, counts, and missing values.
Checking the shape and completeness of the data
Before changing anything, I look at each column's data type and how many values are missing. A blank response might mean someone skipped a question, or that the question didn't apply to them. I don't want to treat every blank as a mistake.

In [5]:
# Write your code here
print("Summary of dataset:", df.info())
print("Missing values:", df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 65437 entries, 0 to 65436
Columns: 114 entries, ResponseId to JobSat
dtypes: float64(13), int64(1), str(100)
memory usage: 195.7 MB
Summary of dataset: None
Missing values: ResponseId                 0
MainBranch                 0
Age                        0
Employment                 0
RemoteWork             10631
                       ...  
JobSatPoints_11        35992
SurveyLength            9255
SurveyEase              9199
ConvertedCompYearly    42002
JobSat                 36311
Length: 114, dtype: int64

There are 65,437 survey responses across 114 columns. The printed None after df.info() is simply the return value of that display function, not a problem with the data.

<h5>2.2 Generate basic statistics for numerical columns.</h5>


Summary statistics help me understand the range and typical values of numerical columns. I pay particular attention to compensation because very large values can pull an average upward. That matters when deciding how to fill missing salary information later.

In [6]:
# Write your code here
print("Basic statistics of columns:", df.describe())

Basic statistics of columns:          ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0.000000         0.00

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


A country might appear under a full official name or a shorter name. If those variations refer to the same place, they should be handled consistently before grouping or comparing responses. Here I first look at the most common country entries, rather than assuming every label needs to change.

In [22]:
# Write your code here
df['Country'].value_counts()

Country
United States of America    11095
Germany                      4947
India                        4231
UK                           3224
Ukraine                      2672
                            ...  
Haiti                           1
Nauru                           1
Chad                            1
Djibouti                        1
Solomon Islands                 1
Name: count, Length: 185, dtype: int64

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


I replace two longer country names with shorter labels: United States of America becomes USA, and United Kingdom of Great Britain and Northern Ireland becomes UK. This makes the output easier to read. It is a targeted replacement, not a full standardization of every country name. In the saved output, USA has 11,095 responses and UK has 3,224.

In [24]:
## Write your code here
df['Country'] = df['Country'].replace({
    'United States of America': 'USA', 
    'United Kingdom of Great Britain and Northern Ireland': 'UK'
})

df['Country'].value_counts().head()

Country
USA        11095
Germany     4947
India       4231
UK          3224
Ukraine     2672
Name: count, dtype: int64

### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


Employment status is text, but some analytical methods need numerical inputs. One-hot encoding creates separate true/false indicators for the employment categories.

The code uses `drop_first=True`, which leaves one category out as a reference. This can be helpful in certain statistical models, but it also means we need to remember that the omitted category is still represented indirectly. If a respondent selected multiple employment statuses, the original categories may be more complex than a simple one-choice question.

In [25]:
## Write your code here
df = pd.get_dummies(df, columns=['Employment'], prefix='Emp', drop_first=True)

### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


I sort the missing-value counts to find the columns with the biggest gaps. In the saved results, several AI-related response columns have especially high counts of missing values. This is a signal to investigate how those survey questions were asked, not an instruction to fill every gap. Some questions may have been optional or shown only to certain respondents.

In [26]:
## Write your code here
df.isnull().sum().sort_values(ascending=False).head()

AINextMuch less integrated    64289
AINextLess integrated         63082
AINextNo change               52939
AINextMuch more integrated    51999
EmbeddedAdmired               48704
dtype: int64

<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


For `ConvertedCompYearly`, I use the median (middle observed value) to replace missing entries. I choose this for the exercise because salary data can be heavily skewed, and the median is less sensitive to extreme amounts than the mean.

**A caution:** The new values are estimates, not salaries the respondents actually reported. This can change the distribution and should be clearly documented before using the data for real compensation analysis.

In [27]:
## Write your code here
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].median())

<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


For the categorical `RemoteWork` field, I use the mode, or most frequently recorded answer. It is a simple way to avoid blank entries when a downstream task requires a complete field. However, this assumes missing responses resemble the most common observed response. If that assumption is wrong, the cleaned data could overstate one work arrangement.

In [28]:
## Write your code here
frequent_value_remote = df['RemoteWork'].mode()[0]
df['RemoteWork'] = df['RemoteWork'].fillna(frequent_value_remote)

### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


Min-Max scaling changes the numerical range of compensation to run from 0 to 1. The smallest observed value becomes 0 and the largest becomes 1; values in between are positioned proportionally. This can help when a model compares variables measured on very different scales. It does not remove extreme salaries: an unusually large maximum can squeeze most other values close together.

In [29]:
## Write your code here
comp_min = df['ConvertedCompYearly'].min()
comp_max = df['ConvertedCompYearly'].max()

df['ConvertedCompYearly_MinMax'] = (df['ConvertedCompYearly'] - comp_min) / (comp_max-comp_min)

<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


The log transformation compresses large compensation values more than small ones. That can make a heavily right-skewed distribution easier to explore.

The `+ 1` allows zero values to be transformed without taking the logarithm of zero. Negative compensation values, if present, would need separate handling. This step creates a new column and keeps the original compensation values available.

In [30]:
## Write your code here
df['ConvertedCompYearly_Log'] = np.log10(df['ConvertedCompYearly'] + 1)

### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


Finally, I group reported years of professional coding experience into categories: Junior (up to 2 years), Mid-Level (over 2 to 5), and Senior (over 5). These are simple thresholds chosen for this lab, not universal job-market definitions.

The code converts non-numeric answers to missing values and then fills them with 0. This means unanswered or non-numeric responses can end up in the Junior group. For a production analysis, I would separate Unknown from genuinely low experience to avoid misleading comparisons.

In [32]:
## Write your code here
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce').fillna(0)

def assign_experience(years):
    if years<=2:
        return 'Junior'
    elif years<=5:
        return 'Mid-Level'
    elif years<=10:
        return 'Senior'
    else:
        return 'Executive'
        
df['ExperienceLevel'] = df['YearsCodePro'].apply(assign_experience)

### Key takeaways
This exercise shows how a sequence of small decisions can make raw survey data easier to analyze: checking data quality, making selected labels consistent, encoding categories, filling some gaps, transforming a skewed numeric variable, and creating a useful experience grouping.

The main lesson for me is that cleaner-looking data is not automatically more accurate data. Each replacement or transformation changes how the dataset can be interpreted.

### Limitations and Improvements
- Check the survey questionnaire and missing-value patterns before deciding whether to impute each field.
- Keep an explicit Unknown category for non-numeric or missing years of experience.
- Compare compensation distributions before and after median imputation and scaling.
- Verify whether employment responses are mutually exclusive before choosing an encoding strategy.
- Add validation checks for unexpected country labels, negative compensation, and any other unusual values.

### Skills demonstrated
- Data exploration: inspecting data types, missingness, and summary statistics
- Data cleaning: targeted label standardization and missing-value imputation
- Feature preparation: one-hot encoding, Min-Max scaling, and logarithmic transformation
- Feature engineering: experience-level categories
- Analytical judgment: documenting assumptions and limitations

### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.
- Encoded categorical variables for analysis.
- Handled missing values using imputation techniques.
- Normalized and transformed numerical data to prepare it for analysis.
- Engineered a new feature to enhance data interpretation.

Copyright © IBM Corporation. All rights reserved.
